# Modelo de moderación — El Aula Informa

Este notebook prueba el modelo que usa la plataforma para detectar contenido ofensivo antes de publicarlo (discurso de odio, agresividad, acoso).

**Modelo:** [RoBERTuito](https://github.com/pysentimiento/pysentimiento) — un modelo de lenguaje tipo transformer (BERT), preentrenado y afinado sobre tuits en español latinoamericano para la tarea de detección de discurso de odio (`hate_speech`).

**Por qué este y no un LLM generativo grande (GPT/Llama):** un LLM generativo necesita GPU y hosting de paga para responder en segundos; RoBERTuito corre en CPU gratis y es justo la arquitectura que se usa en producción para moderación de contenido en tiempo real (es el mismo principio que Perspective API de Google o el endpoint de moderación de OpenAI: un modelo de lenguaje afinado para clasificar, no para generar texto).

In [ ]:
%pip install -q pysentimiento torch

In [ ]:
from pysentimiento import create_analyzer

analizador = create_analyzer(task="hate_speech", lang="es")

## Probando con ejemplos

Textos parecidos a lo que se publicaría en el muro de la comunidad (avisos, testimonios, propuestas). El modelo devuelve una probabilidad por etiqueta: `hateful` (odio), `aggressive` (agresivo), `targeted` (dirigido a alguien).

In [ ]:
ejemplos = [
    "Se cancela la asamblea de mañana por falta de quórum.",
    "Llevo tres días sin poder inscribirme por el paro, esto ya es el colmo.",
    "Propongo que la próxima asamblea sea en la explanada principal.",
    "Eres un inútil, no sirves para nada y deberían correrte.",
]

for texto in ejemplos:
    r = analizador.predict(texto)
    etiqueta, confianza = max(r.probas.items(), key=lambda kv: kv[1])
    print(f"{texto!r}\n  -> {etiqueta} ({confianza:.2f})  todas: {r.probas}\n")

## Umbral de decisión

El servicio (`app.py`) usa un umbral (por defecto 0.5): si la etiqueta más alta supera ese valor, el texto se rechaza con un motivo (`discurso_de_odio`, `lenguaje_ofensivo` o `acoso_u_ofensa`). Aquí se puede probar cómo cambia el resultado según el umbral, para elegir uno que balancee falsos positivos (bloquear texto normal) contra falsos negativos (dejar pasar contenido ofensivo).

In [ ]:
def moderar(texto: str, umbral: float = 0.5):
    r = analizador.predict(texto)
    etiqueta, confianza = max(r.probas.items(), key=lambda kv: kv[1])
    ofensivo = confianza >= umbral and etiqueta in ("hateful", "aggressive", "targeted")
    return {"ofensivo": ofensivo, "motivo": etiqueta if ofensivo else None, "confianza": round(confianza, 3)}


moderar("Eres un inútil, no sirves para nada y deberían correrte.")